In [ ]:
import json
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Use the fitness.db from the backend
db_path = os.path.join(os.path.dirname(__file__), "..", "backend", "fitness.db")

def get_user_profiles():
    """Fetch all user profiles from the database."""
    if not os.path.exists(db_path):
        print("⚠️ Database not found. Run the server first to create it.")
        return []

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    cur.execute("""
        SELECT u.id, u.username, u.email,
               ui.age, ui.gender, ui.goal, ui.fitness_level, ui.profile_active
        FROM users u
        LEFT JOIN userInfo ui ON u.id = ui.user_id
    """)

    profiles = [dict(row) for row in cur.fetchall()]
    conn.close()
    return profiles

# Load profiles from database
profiles_list = get_user_profiles()

if not profiles_list:
    print("⚠️ No user profiles found.")
else:
    print("📌 Loaded User Profiles:")
    display(profiles_list)

    # Convert to DataFrame for clean display
    df = pd.DataFrame(profiles_list)
    print("\n📄 Profiles as DataFrame:")
    display(df)

    # -----------------------------------------------
    # VISUAL 1: BAR CHART OF NUMERIC USER FEATURES
    # -----------------------------------------------
    if 'age' in df.columns and df['age'].notna().any():
        plt.figure(figsize=(6, 4))
        sns.barplot(x=df.index, y=df['age'])
        plt.title("User Ages")
        plt.ylabel("Age")
        plt.show()
    else:
        print("⚠️ No age data available for visualization.")

    # -----------------------------------------------
    # VISUAL 2: GOAL DISTRIBUTION
    # -----------------------------------------------
    if 'goal' in df.columns and df['goal'].notna().any():
        plt.figure(figsize=(6, 4))
        df['goal'].value_counts().plot(kind='bar')
        plt.title("Fitness Goals Distribution")
        plt.ylabel("Count")
        plt.xticks(rotation=45)
        plt.show()
    else:
        print("⚠️ No goal data available for visualization.")